# Phase 3B — Retrieval Evaluation

This notebook evaluates three retrieval approaches using the same Phase 3A
ground-truth dataset:

1. Minsearch text retrieval
2. Vector retrieval
3. Hybrid retrieval

The evaluation metrics are:

- Hit Rate@5
- MRR@5
- Average retrieval latency

For every question, the expected relevant document is identified by `chunk_id`.

In [1]:
import json
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CHUNKS_PATH = PROJECT_ROOT / "data" / "chunks.parquet"
GROUND_TRUTH_PATH = PROJECT_ROOT / "data" / "evaluation" / "ground-truth.json"

print("Project root:", PROJECT_ROOT)
print("Chunks path exists:", CHUNKS_PATH.exists())
print("Ground-truth path exists:", GROUND_TRUTH_PATH.exists())

Project root: /workspaces/pm-playbook
Chunks path exists: True
Ground-truth path exists: True


In [2]:
with GROUND_TRUTH_PATH.open(encoding="utf-8") as f:
    ground_truth = json.load(f)

chunks_df = pd.read_parquet(CHUNKS_PATH)
ground_truth_df = pd.DataFrame(ground_truth)

print("Ground-truth shape:", ground_truth_df.shape)
print("Chunks shape:", chunks_df.shape)

print("\nRequired ground-truth columns:")
required_ground_truth_columns = {"question", "chunk_id"}
print(required_ground_truth_columns.issubset(ground_truth_df.columns))

print("\nUnique ground-truth questions:", ground_truth_df["question"].nunique())
print("Unique ground-truth chunk IDs:", ground_truth_df["chunk_id"].nunique())
print("Duplicate corpus chunk IDs:", chunks_df["chunk_id"].duplicated().sum())

missing_chunk_ids = set(ground_truth_df["chunk_id"]) - set(chunks_df["chunk_id"])
print("Missing ground-truth chunk IDs:", len(missing_chunk_ids))

Ground-truth shape: (180, 19)
Chunks shape: (50910, 15)

Required ground-truth columns:
True

Unique ground-truth questions: 180
Unique ground-truth chunk IDs: 180
Duplicate corpus chunk IDs: 0
Missing ground-truth chunk IDs: 0


In [3]:
TEXT_FIELDS = [
    "text",
    "episode_title",
    "guest",
    "speaker_name",
]

KEYWORD_FIELDS = [
    "chunk_id",
    "episode_id",
    "guest",
    "speaker_name",
    "video_id",
    "is_sponsor_read",
]

DEFAULT_BOOSTS = {
    "text": 1.0,
    "episode_title": 2.0,
    "guest": 3.0,
    "speaker_name": 2.0,
}

baseline_df = chunks_df[
    ~chunks_df["is_sponsor_read"].fillna(False)
].copy()

baseline_df = baseline_df[
    baseline_df["text"].fillna("").str.strip().ne("")
].copy()

baseline_df = baseline_df[
    baseline_df["word_count"].fillna(0).ge(20)
].copy()

baseline_df = baseline_df.where(pd.notna(baseline_df), None)

baseline_documents = baseline_df.to_dict(orient="records")

print("Original corpus rows:", len(chunks_df))
print("Baseline searchable rows:", len(baseline_df))
print("Rows excluded:", len(chunks_df) - len(baseline_df))
print("Documents prepared:", len(baseline_documents))

missing_ground_truth_ids = (
    set(ground_truth_df["chunk_id"]) - set(baseline_df["chunk_id"])
)

print("Ground-truth chunks excluded from baseline:", len(missing_ground_truth_ids))

Original corpus rows: 50910
Baseline searchable rows: 38760
Rows excluded: 12150
Documents prepared: 38760
Ground-truth chunks excluded from baseline: 0


In [4]:
from minsearch import Index

text_index = Index(
    text_fields=TEXT_FIELDS,
    keyword_fields=KEYWORD_FIELDS,
)

text_index.fit(baseline_documents)

print("Minsearch index built.")
print("Indexed documents:", len(baseline_documents))

Minsearch index built.
Indexed documents: 38760


In [5]:
sample_record = ground_truth_df.iloc[0]

sample_question = sample_record["question"]
expected_chunk_id = sample_record["chunk_id"]

sample_results = text_index.search(
    query=sample_question,
    boost_dict=DEFAULT_BOOSTS,
    num_results=5,
)

print("Question:", sample_question)
print("Expected chunk ID:", expected_chunk_id)
print("\nTop 5 retrieved chunk IDs:")

for rank, result in enumerate(sample_results, start=1):
    marker = "MATCH" if result["chunk_id"] == expected_chunk_id else ""
    print(
        f"{rank}. {result['chunk_id']} "
        f"| guest={result['guest']} "
        f"| {marker}"
    )

Question: What challenge do many organizations face regarding AI adoption among their team members?
Expected chunk ID: 248716fafa12abd5c713bd05bf772ce9

Top 5 retrieved chunk IDs:
1. d539e11c6a47ae1e0e519e90b19d4869 | guest=Jason M Lemkin | 
2. 454b7e22c58d6a4ed260d72433142243 | guest=Jason M Lemkin | 
3. d5e9efe2640d72458a10f277cc725a46 | guest=Jason M Lemkin | 
4. 607ae94dbde80507dd2e1f16176bab72 | guest=Jason M Lemkin | 
5. 54327358639386b52a50ca769b44737f | guest=Jason M Lemkin | 


In [6]:
expected_row = baseline_df[
    baseline_df["chunk_id"] == expected_chunk_id
].iloc[0]

print("EXPECTED SOURCE")
print("Guest:", expected_row["guest"])
print("Title:", expected_row["episode_title"])
print("Text:")
print(expected_row["text"])

print("\n" + "=" * 100)

for rank, result in enumerate(sample_results, start=1):
    print(f"\nRESULT {rank}")
    print("Chunk ID:", result["chunk_id"])
    print("Guest:", result["guest"])
    print("Title:", result["episode_title"])
    print("Text:")
    print(result["text"][:700])
    print("-" * 100)

EXPECTED SOURCE
Guest: Dan Shipper
Title: The AI-native startup: 5 products, 7-figure revenue, 100% AI-written code. | Dan Shipper (Every)
Text:
And I think for us it's a little easier because everybody inside the org is very AI-first and just wants to go do it. We don't have anyone really who's like, "I don't know. I don't really want to do this." And that's a whole different challenge, which I think a lot of organizations face, but there's always a problem of getting people to use it.


RESULT 1
Chunk ID: d539e11c6a47ae1e0e519e90b19d4869
Guest: Jason M Lemkin
Title: We replaced our sales team with 20 AI agents—here’s what happened next | Jason Lemkin (SaaStr)
Text:
It's absolutely my pleasure. I feel like you're the kind of guy that I knew would be on this podcast eventually, and I'm glad that we're finally doing this. I also feel like we can go in so many directions. I feel like you have so many insights on so many parts of building a business, especially a B2B business, but I thoug

In [7]:
import numpy as np
EMBEDDINGS_DIR = PROJECT_ROOT / "data" / "embeddings"

VECTOR_EMBEDDINGS_PATH = (
    EMBEDDINGS_DIR / "all-MiniLM-L6-v2-text-embeddings.npy"
)
VECTOR_CHUNK_IDS_PATH = (
    EMBEDDINGS_DIR / "all-MiniLM-L6-v2-text-chunk-ids.json"
)

vector_embeddings = np.load(
    VECTOR_EMBEDDINGS_PATH,
    mmap_mode="r",
)

with VECTOR_CHUNK_IDS_PATH.open(encoding="utf-8") as f:
    vector_chunk_ids = json.load(f)

print("Embedding shape:", vector_embeddings.shape)
print("Embedding dtype:", vector_embeddings.dtype)
print("Chunk ID count:", len(vector_chunk_ids))
print(
    "Aligned lengths:",
    vector_embeddings.shape[0] == len(vector_chunk_ids),
)

Embedding shape: (38760, 384)
Embedding dtype: float32
Chunk ID count: 38760
Aligned lengths: True


In [8]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)
print("Embedding dimension:", embedding_model.get_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded: all-MiniLM-L6-v2
Embedding dimension: 384


In [9]:
sample_query_embedding = embedding_model.encode(
    [sample_question],
    normalize_embeddings=True,
    show_progress_bar=False,
    convert_to_numpy=True,
)[0]

sample_scores = vector_embeddings @ sample_query_embedding

top_indices = np.argsort(sample_scores)[::-1][:5]

print("Question:", sample_question)
print("Expected chunk ID:", expected_chunk_id)
print("\nTop 5 vector results:")

for rank, index_position in enumerate(top_indices, start=1):
    chunk_id = vector_chunk_ids[index_position]
    row = baseline_df.iloc[index_position]
    marker = "MATCH" if chunk_id == expected_chunk_id else ""

    print(
        f"{rank}. {chunk_id} "
        f"| score={sample_scores[index_position]:.4f} "
        f"| guest={row['guest']} "
        f"| {marker}"
    )

Question: What challenge do many organizations face regarding AI adoption among their team members?
Expected chunk ID: 248716fafa12abd5c713bd05bf772ce9

Top 5 vector results:
1. b34ca1bc7394e6d0ce4ba7329a2ca8cc | score=0.6968 | guest=Brian Balfour | 
2. 3919b8111cb68b480f54791d29a67c01 | score=0.6387 | guest=Asha Sharma | 
3. 71c06b9eeaf043e049b8113abc366766 | score=0.6212 | guest=Elena Verna 4.0 | 
4. 4ffa3464512f220620a45e79d2193e64 | score=0.6156 | guest=Aishwarya Naresh Reganti + Kiriti Badam | 
5. 507fd2c0fa77b8494e56b8f200512674 | score=0.6141 | guest=Dan Shipper | 


In [10]:
expected_index = vector_chunk_ids.index(expected_chunk_id)
expected_score = sample_scores[expected_index]

ranked_indices = np.argsort(sample_scores)[::-1]
expected_rank = int(np.where(ranked_indices == expected_index)[0][0]) + 1

print("Expected chunk score:", float(expected_score))
print("Expected chunk rank:", expected_rank)
print("Retrieved in top 5:", expected_rank <= 5)

Expected chunk score: 0.5454161167144775
Expected chunk rank: 68
Retrieved in top 5: False


In [11]:
def hit_rate_at_k(
    retrieved_ids: list[str],
    expected_id: str,
    k: int = 5,
) -> float:
    """
    Return 1.0 when the expected document appears in the top-k results,
    otherwise return 0.0.
    """
    return float(expected_id in retrieved_ids[:k])


def reciprocal_rank_at_k(
    retrieved_ids: list[str],
    expected_id: str,
    k: int = 5,
) -> float:
    """
    Return the reciprocal rank of the expected document within the top-k
    results. Return 0.0 when it is not retrieved.
    """
    for rank, chunk_id in enumerate(retrieved_ids[:k], start=1):
        if chunk_id == expected_id:
            return 1.0 / rank

    return 0.0

In [12]:
test_ids = ["chunk-a", "chunk-b", "chunk-c", "chunk-d", "chunk-e"]

print("Hit at rank 1:", hit_rate_at_k(test_ids, "chunk-a", k=5))
print("MRR at rank 1:", reciprocal_rank_at_k(test_ids, "chunk-a", k=5))

print("Hit at rank 3:", hit_rate_at_k(test_ids, "chunk-c", k=5))
print("MRR at rank 3:", reciprocal_rank_at_k(test_ids, "chunk-c", k=5))

print("Missing hit:", hit_rate_at_k(test_ids, "chunk-z", k=5))
print("Missing MRR:", reciprocal_rank_at_k(test_ids, "chunk-z", k=5))

Hit at rank 1: 1.0
MRR at rank 1: 1.0
Hit at rank 3: 1.0
MRR at rank 3: 0.3333333333333333
Missing hit: 0.0
Missing MRR: 0.0


In [13]:
## — Create the Minsearch evaluation function

def evaluate_text_retrieval(
    evaluation_df: pd.DataFrame,
    index: Index,
    *,
    k: int = 5,
) -> tuple[pd.DataFrame, dict[str, float]]:
    """
    Evaluate Minsearch retrieval against the ground-truth chunk IDs.
    """
    rows = []

    for record in evaluation_df.itertuples(index=False):
        started_at = time.perf_counter()

        results = index.search(
            query=record.question,
            boost_dict=DEFAULT_BOOSTS,
            num_results=k,
        )

        latency_ms = (time.perf_counter() - started_at) * 1000

        retrieved_ids = [result["chunk_id"] for result in results]

        rows.append(
            {
                "question_id": record.question_id,
                "question": record.question,
                "expected_chunk_id": record.chunk_id,
                "retrieved_chunk_ids": retrieved_ids,
                "hit_at_5": hit_rate_at_k(
                    retrieved_ids,
                    record.chunk_id,
                    k=k,
                ),
                "reciprocal_rank_at_5": reciprocal_rank_at_k(
                    retrieved_ids,
                    record.chunk_id,
                    k=k,
                ),
                "latency_ms": latency_ms,
            }
        )

    results_df = pd.DataFrame(rows)

    summary = {
        "questions": float(len(results_df)),
        "hit_rate_at_5": results_df["hit_at_5"].mean(),
        "mrr_at_5": results_df["reciprocal_rank_at_5"].mean(),
        "average_latency_ms": results_df["latency_ms"].mean(),
    }

    return results_df, summary

In [14]:
## testing the minsearch evaluation function

text_test_results_df, text_test_summary = evaluate_text_retrieval(
    ground_truth_df.head(3),
    text_index,
    k=5,
)

print(text_test_summary)

text_test_results_df[
    [
        "question_id",
        "hit_at_5",
        "reciprocal_rank_at_5",
        "latency_ms",
    ]
]

{'questions': 3.0, 'hit_rate_at_5': np.float64(0.6666666666666666), 'mrr_at_5': np.float64(0.3333333333333333), 'average_latency_ms': np.float64(84.8684030000489)}


,question_id,hit_at_5,reciprocal_rank_at_5,latency_ms
0,question-248716fafa12abd5c713bd05bf772ce9,0.0,0.0,91.739673
1,question-b6619b3c795e8067df765b4de16e3e13,1.0,0.5,75.326085
2,question-e6e7f0d1e36a399dc842a84f8b7616ae,1.0,0.5,87.539451


In [15]:
## running full search evaluation

text_results_df, text_summary = evaluate_text_retrieval(
    ground_truth_df,
    text_index,
    k=5,
)

print("Minsearch text baseline")
print(f"Questions: {int(text_summary['questions'])}")
print(f"Hit Rate@5: {text_summary['hit_rate_at_5']:.4f}")
print(f"MRR@5: {text_summary['mrr_at_5']:.4f}")
print(
    "Average latency: "
    f"{text_summary['average_latency_ms']:.2f} ms"
)

Minsearch text baseline
Questions: 180
Hit Rate@5: 0.4167
MRR@5: 0.2706
Average latency: 47.74 ms


In [16]:
print("\nEvaluation rows:", len(text_results_df))
print(
    "Rows with five retrieved IDs:",
    text_results_df["retrieved_chunk_ids"].apply(len).eq(5).sum(),
)
print(
    "Missing metric values:",
    text_results_df[
        [
            "hit_at_5",
            "reciprocal_rank_at_5",
            "latency_ms",
        ]
    ].isna().sum().sum(),
)


Evaluation rows: 180
Rows with five retrieved IDs: 180
Missing metric values: 0


In [17]:
## Save the baseline results in the notebook session

text_metrics = {
    "retrieval_method": "minsearch_text",
    "questions": int(text_summary["questions"]),
    "hit_rate_at_5": text_summary["hit_rate_at_5"],
    "mrr_at_5": text_summary["mrr_at_5"],
    "average_latency_ms": text_summary["average_latency_ms"],
}

metrics_rows = [text_metrics]

metrics_df = pd.DataFrame(metrics_rows)

metrics_df


,retrieval_method,questions,hit_rate_at_5,mrr_at_5,average_latency_ms
0,minsearch_text,180,0.416667,0.270556,47.736135


In [18]:

print("Successful top-5 hits:", int(text_results_df["hit_at_5"].sum()))
print("Top-5 misses:", int((text_results_df["hit_at_5"] == 0).sum()))

print("\nExpected total:")
print(
    int(text_results_df["hit_at_5"].sum())
    + int((text_results_df["hit_at_5"] == 0).sum())
)

Successful top-5 hits: 75
Top-5 misses: 105

Expected total:
180


In [19]:
## loading model

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)
print("Embedding dimension:", embedding_model.get_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded: all-MiniLM-L6-v2
Embedding dimension: 384


In [20]:
sample_texts = baseline_df["text"].head(3).tolist()

sample_embeddings = embedding_model.encode(
    sample_texts,
    normalize_embeddings=True,
    show_progress_bar=False,
)

print("Sample embedding shape:", sample_embeddings.shape)
print("Embedding dtype:", sample_embeddings.dtype)
print("First vector norm:", float((sample_embeddings[0] ** 2).sum() ** 0.5))

Sample embedding shape: (3, 384)
Embedding dtype: float32
First vector norm: 1.0


In [21]:
import numpy as np

sample_query_embedding = embedding_model.encode(
    [sample_question],
    normalize_embeddings=True,
    show_progress_bar=False,
    convert_to_numpy=True,
)[0]

sample_scores = vector_embeddings @ sample_query_embedding

top_indices = np.argsort(sample_scores)[::-1][:5]

print("Question:", sample_question)
print("Expected chunk ID:", expected_chunk_id)
print("\nTop 5 vector results:")

for rank, index_position in enumerate(top_indices, start=1):
    chunk_id = vector_chunk_ids[index_position]
    row = baseline_df.iloc[index_position]
    marker = "MATCH" if chunk_id == expected_chunk_id else ""

    print(
        f"{rank}. {chunk_id} "
        f"| score={sample_scores[index_position]:.4f} "
        f"| guest={row['guest']} "
        f"| {marker}"
    )

Question: What challenge do many organizations face regarding AI adoption among their team members?
Expected chunk ID: 248716fafa12abd5c713bd05bf772ce9

Top 5 vector results:
1. b34ca1bc7394e6d0ce4ba7329a2ca8cc | score=0.6968 | guest=Brian Balfour | 
2. 3919b8111cb68b480f54791d29a67c01 | score=0.6387 | guest=Asha Sharma | 
3. 71c06b9eeaf043e049b8113abc366766 | score=0.6212 | guest=Elena Verna 4.0 | 
4. 4ffa3464512f220620a45e79d2193e64 | score=0.6156 | guest=Aishwarya Naresh Reganti + Kiriti Badam | 
5. 507fd2c0fa77b8494e56b8f200512674 | score=0.6141 | guest=Dan Shipper | 


In [22]:
def evaluate_vector_retrieval(
    evaluation_df: pd.DataFrame,
    model: SentenceTransformer,
    document_embeddings: np.ndarray,
    chunk_ids: list[str],
    *,
    k: int = 5,
) -> tuple[pd.DataFrame, dict[str, float]]:
    """
    Evaluate normalized embedding retrieval against ground-truth chunk IDs.

    Latency includes query embedding generation and similarity search,
    but excludes the one-time corpus embedding build.
    """
    rows = []

    for record in evaluation_df.itertuples(index=False):
        started_at = time.perf_counter()

        query_embedding = model.encode(
            [record.question],
            normalize_embeddings=True,
            show_progress_bar=False,
            convert_to_numpy=True,
        )[0]

        scores = document_embeddings @ query_embedding
        top_indices = np.argsort(scores)[::-1][:k]
        retrieved_ids = [chunk_ids[index] for index in top_indices]

        latency_ms = (time.perf_counter() - started_at) * 1000

        rows.append(
            {
                "question_id": record.question_id,
                "question": record.question,
                "expected_chunk_id": record.chunk_id,
                "retrieved_chunk_ids": retrieved_ids,
                "retrieved_scores": [
                    float(scores[index]) for index in top_indices
                ],
                "hit_at_5": hit_rate_at_k(
                    retrieved_ids,
                    record.chunk_id,
                    k=k,
                ),
                "reciprocal_rank_at_5": reciprocal_rank_at_k(
                    retrieved_ids,
                    record.chunk_id,
                    k=k,
                ),
                "latency_ms": latency_ms,
            }
        )

    results_df = pd.DataFrame(rows)

    summary = {
        "questions": float(len(results_df)),
        "hit_rate_at_5": results_df["hit_at_5"].mean(),
        "mrr_at_5": results_df["reciprocal_rank_at_5"].mean(),
        "average_latency_ms": results_df["latency_ms"].mean(),
    }

    return results_df, summary

In [23]:
vector_test_results_df, vector_test_summary = evaluate_vector_retrieval(
    ground_truth_df.head(3),
    embedding_model,
    vector_embeddings,
    vector_chunk_ids,
    k=5,
)

print(vector_test_summary)

vector_test_results_df[
    [
        "question_id",
        "hit_at_5",
        "reciprocal_rank_at_5",
        "latency_ms",
    ]
]

{'questions': 3.0, 'hit_rate_at_5': np.float64(0.3333333333333333), 'mrr_at_5': np.float64(0.16666666666666666), 'average_latency_ms': np.float64(39.73196400003568)}


,question_id,hit_at_5,reciprocal_rank_at_5,latency_ms
0,question-248716fafa12abd5c713bd05bf772ce9,0.0,0.0,22.995749
1,question-b6619b3c795e8067df765b4de16e3e13,0.0,0.0,24.219698
2,question-e6e7f0d1e36a399dc842a84f8b7616ae,1.0,0.5,71.980445


In [24]:
vector_results_df, vector_summary = evaluate_vector_retrieval(
    ground_truth_df,
    embedding_model,
    vector_embeddings,
    vector_chunk_ids,
    k=5,
)

print("Vector retrieval baseline")
print(f"Questions: {int(vector_summary['questions'])}")
print(f"Hit Rate@5: {vector_summary['hit_rate_at_5']:.4f}")
print(f"MRR@5: {vector_summary['mrr_at_5']:.4f}")
print(
    "Average latency: "
    f"{vector_summary['average_latency_ms']:.2f} ms"
)

print("\nEvaluation rows:", len(vector_results_df))
print(
    "Rows with five retrieved IDs:",
    vector_results_df["retrieved_chunk_ids"].apply(len).eq(5).sum(),
)
print(
    "Missing metric values:",
    vector_results_df[
        [
            "hit_at_5",
            "reciprocal_rank_at_5",
            "latency_ms",
        ]
    ].isna().sum().sum(),
)

Vector retrieval baseline
Questions: 180
Hit Rate@5: 0.4444
MRR@5: 0.3124
Average latency: 29.91 ms

Evaluation rows: 180
Rows with five retrieved IDs: 180
Missing metric values: 0


In [25]:
vector_metrics = {
    "retrieval_method": "vector_text_all-MiniLM-L6-v2",
    "questions": int(vector_summary["questions"]),
    "hit_rate_at_5": vector_summary["hit_rate_at_5"],
    "mrr_at_5": vector_summary["mrr_at_5"],
    "average_latency_ms": vector_summary["average_latency_ms"],
}

metrics_rows = [
    text_metrics,
    vector_metrics,
]

metrics_df = pd.DataFrame(metrics_rows)

metrics_df

,retrieval_method,questions,hit_rate_at_5,mrr_at_5,average_latency_ms
0,minsearch_text,180,0.416667,0.270556,47.736135
1,vector_text_all-MiniLM-L6-v2,180,0.444444,0.312407,29.910730


In [26]:
print(
    "Vector successful top-5 hits:",
    int(vector_results_df["hit_at_5"].sum()),
)
print(
    "Vector top-5 misses:",
    int((vector_results_df["hit_at_5"] == 0).sum()),
)

Vector successful top-5 hits: 80
Vector top-5 misses: 100


In [27]:
def reciprocal_rank_fusion(
    text_chunk_ids: list[str],
    vector_chunk_ids: list[str],
    *,
    rrf_k: int = 60,
    limit: int = 5,
) -> list[str]:
    """
    Combine two ranked chunk-ID lists using Reciprocal Rank Fusion.
    """
    fused_scores: dict[str, float] = {}

    for ranked_ids in (text_chunk_ids, vector_chunk_ids):
        for rank, chunk_id in enumerate(ranked_ids, start=1):
            fused_scores[chunk_id] = fused_scores.get(chunk_id, 0.0) + (
                1.0 / (rrf_k + rank)
            )

    ranked_chunk_ids = sorted(
        fused_scores,
        key=fused_scores.get,
        reverse=True,
    )

    return ranked_chunk_ids[:limit]

In [28]:
test_text_ids = ["a", "b", "c", "d", "e"]
test_vector_ids = ["c", "a", "f", "g", "h"]

test_hybrid_ids = reciprocal_rank_fusion(
    test_text_ids,
    test_vector_ids,
    rrf_k=60,
    limit=5,
)

print("Hybrid ranking:", test_hybrid_ids)
print("Result count:", len(test_hybrid_ids))
print("Unique results:", len(set(test_hybrid_ids)))

Hybrid ranking: ['a', 'c', 'b', 'f', 'd']
Result count: 5
Unique results: 5


In [29]:
## Add the hybrid evaluation function

def evaluate_hybrid_retrieval(
    evaluation_df: pd.DataFrame,
    text_index: Index,
    model: SentenceTransformer,
    document_embeddings: np.ndarray,
    chunk_ids: list[str],
    *,
    text_candidates: int = 20,
    vector_candidates: int = 20,
    k: int = 5,
    rrf_k: int = 60,
) -> tuple[pd.DataFrame, dict[str, float]]:
    """
    Evaluate hybrid retrieval using Reciprocal Rank Fusion over
    Minsearch and vector candidate rankings.
    """
    rows = []

    for record in evaluation_df.itertuples(index=False):
        started_at = time.perf_counter()

        text_results = text_index.search(
            query=record.question,
            boost_dict=DEFAULT_BOOSTS,
            num_results=text_candidates,
        )
        text_ids = [result["chunk_id"] for result in text_results]

        query_embedding = model.encode(
            [record.question],
            normalize_embeddings=True,
            show_progress_bar=False,
            convert_to_numpy=True,
        )[0]

        vector_scores = document_embeddings @ query_embedding
        vector_top_indices = np.argsort(vector_scores)[::-1][:vector_candidates]
        vector_ids = [chunk_ids[index] for index in vector_top_indices]

        retrieved_ids = reciprocal_rank_fusion(
            text_ids,
            vector_ids,
            rrf_k=rrf_k,
            limit=k,
        )

        latency_ms = (time.perf_counter() - started_at) * 1000

        rows.append(
            {
                "question_id": record.question_id,
                "question": record.question,
                "expected_chunk_id": record.chunk_id,
                "retrieved_chunk_ids": retrieved_ids,
                "text_candidate_ids": text_ids,
                "vector_candidate_ids": vector_ids,
                "hit_at_5": hit_rate_at_k(
                    retrieved_ids,
                    record.chunk_id,
                    k=k,
                ),
                "reciprocal_rank_at_5": reciprocal_rank_at_k(
                    retrieved_ids,
                    record.chunk_id,
                    k=k,
                ),
                "latency_ms": latency_ms,
            }
        )

    results_df = pd.DataFrame(rows)

    summary = {
        "questions": float(len(results_df)),
        "hit_rate_at_5": results_df["hit_at_5"].mean(),
        "mrr_at_5": results_df["reciprocal_rank_at_5"].mean(),
        "average_latency_ms": results_df["latency_ms"].mean(),
    }

    return results_df, summary

In [30]:
## 3 question cell

hybrid_test_results_df, hybrid_test_summary = evaluate_hybrid_retrieval(
    ground_truth_df.head(3),
    text_index,
    embedding_model,
    vector_embeddings,
    vector_chunk_ids,
    text_candidates=20,
    vector_candidates=20,
    k=5,
    rrf_k=60,
)

print(hybrid_test_summary)

hybrid_test_results_df[
    [
        "question_id",
        "hit_at_5",
        "reciprocal_rank_at_5",
        "latency_ms",
    ]
]

{'questions': 3.0, 'hit_rate_at_5': np.float64(0.6666666666666666), 'mrr_at_5': np.float64(0.39999999999999997), 'average_latency_ms': np.float64(78.07794766677034)}


,question_id,hit_at_5,reciprocal_rank_at_5,latency_ms
0,question-248716fafa12abd5c713bd05bf772ce9,0.0,0.0,74.180668
1,question-b6619b3c795e8067df765b4de16e3e13,1.0,0.2,80.048904
2,question-e6e7f0d1e36a399dc842a84f8b7616ae,1.0,1.0,80.004271


In [31]:
hybrid_results_df, hybrid_summary = evaluate_hybrid_retrieval(
    ground_truth_df,
    text_index,
    embedding_model,
    vector_embeddings,
    vector_chunk_ids,
    text_candidates=20,
    vector_candidates=20,
    k=5,
    rrf_k=60,
)

print("Hybrid retrieval baseline")
print(f"Questions: {int(hybrid_summary['questions'])}")
print(f"Hit Rate@5: {hybrid_summary['hit_rate_at_5']:.4f}")
print(f"MRR@5: {hybrid_summary['mrr_at_5']:.4f}")
print(
    "Average latency: "
    f"{hybrid_summary['average_latency_ms']:.2f} ms"
)

print("\nEvaluation rows:", len(hybrid_results_df))
print(
    "Rows with five retrieved IDs:",
    hybrid_results_df["retrieved_chunk_ids"].apply(len).eq(5).sum(),
)
print(
    "Missing metric values:",
    hybrid_results_df[
        [
            "hit_at_5",
            "reciprocal_rank_at_5",
            "latency_ms",
        ]
    ].isna().sum().sum(),
)

Hybrid retrieval baseline
Questions: 180
Hit Rate@5: 0.5722
MRR@5: 0.4075
Average latency: 89.54 ms

Evaluation rows: 180
Rows with five retrieved IDs: 180
Missing metric values: 0


In [32]:
hybrid_metrics = {
    "retrieval_method": "hybrid_rrf",
    "questions": int(hybrid_summary["questions"]),
    "hit_rate_at_5": hybrid_summary["hit_rate_at_5"],
    "mrr_at_5": hybrid_summary["mrr_at_5"],
    "average_latency_ms": hybrid_summary["average_latency_ms"],
}

metrics_rows = [
    text_metrics,
    vector_metrics,
    hybrid_metrics,
]

metrics_df = pd.DataFrame(metrics_rows)

metrics_df["successful_hits_at_5"] = (
    metrics_df["hit_rate_at_5"] * metrics_df["questions"]
).round().astype(int)

metrics_df["top_5_misses"] = (
    metrics_df["questions"] - metrics_df["successful_hits_at_5"]
)

metrics_df

,retrieval_method,questions,hit_rate_at_5,mrr_at_5,average_latency_ms,successful_hits_at_5,top_5_misses
0,minsearch_text,180,0.416667,0.270556,47.736135,75,105
1,vector_text_all-MiniLM-L6-v2,180,0.444444,0.312407,29.910730,80,100
2,hybrid_rrf,180,0.572222,0.407500,89.544250,103,77


In [33]:
print(
    "Hybrid successful top-5 hits:",
    int(hybrid_results_df["hit_at_5"].sum()),
)
print(
    "Hybrid top-5 misses:",
    int((hybrid_results_df["hit_at_5"] == 0).sum()),
)

Hybrid successful top-5 hits: 103
Hybrid top-5 misses: 77
